# Modelagem e treino — câncer de mama (maligno vs benigno)

Treino de **Regressão Logística** e **Árvore de Decisão** apenas no conjunto de treinamento — sem avaliar o teste.

- **Entrada:** `outputs/artifacts/dados_preprocessados.joblib` (notebook 03)
- **Alvo:** 1 = maligno, 0 = benigno
- **Artefato:** `outputs/artifacts/modelos_treinados.joblib`
- **Próximo:** métricas e escolha do modelo ficam no notebook 05

## 1 - Setup e carregamento de artefato

In [1]:
# Célula comum (v2) + imports de modelagem
from pathlib import Path

import joblib
from sklearn.pipeline import Pipeline

# Modelos que serão usados:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

# Encontra a raiz do repo estando em /notebooks ou na raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PASTA_ENTRADA = RAIZ / "src/db/outputs/03-preprocessamento"
PASTA_BASE = RAIZ / "src/db/outputs/04-modelagem-treino/"
PASTA_FIGURAS = PASTA_BASE / "figures"
PASTA_METRICAS = PASTA_BASE / "metrics"
PASTA_ARTEFATOS = PASTA_BASE / "artifacts"
CAMINHO_ARTEFATO_03 = PASTA_ENTRADA / "artifacts" / "dados_preprocessados.joblib"

for pasta in (PASTA_FIGURAS, PASTA_METRICAS, PASTA_ARTEFATOS):
    pasta.mkdir(parents=True, exist_ok=True)

### 1.1 - Carregamento de artefato do notebook 03

In [2]:
# Carrega split + preprocessador — sem refazer o notebook 03
if not CAMINHO_ARTEFATO_03.exists():
    raise FileNotFoundError("Execute o notebook 03 primeiro.")

pacote = joblib.load(CAMINHO_ARTEFATO_03)
atributos_treino = pacote["atributos_treino"] # Features para treinamento
atributos_teste = pacote["atributos_teste"] # Features para teste
rotulo_treino = pacote["rotulo_treino"] # Diagnósticos correspondentes ao treino
rotulo_teste = pacote["rotulo_teste"] # Diagnósticos correspondentes ao teste
colunas_atributos = pacote["colunas_atributos"]
preprocessador = pacote["preprocessador"]

print(f"Artefato 03: {CAMINHO_ARTEFATO_03}")
print(f"Treino: {len(atributos_treino)} | Teste: {len(atributos_teste)} (guardado, sem avaliar)")
print(f"Atributos: {len(colunas_atributos)}")

Artefato 03: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\03-preprocessamento\artifacts\dados_preprocessados.joblib
Treino: 455 | Teste: 114 (guardado, sem avaliar)
Atributos: 30


## 2 - Justificativa dos modelos

| Modelo | Por quê |
|--------|---------|
| Regressão Logística | Por ser util para triagem por fornecer **probabilidades** de malignidade |
| Árvore de Decisão | Obter relações **não lineares** e permite inspecionar **importância de atributos** |

## 3 - Treinar pipelines (somente treino)

In [3]:
SEMENTE = 42

# Esse pipeline irá garantir a execução na ordem de preprocessar e posteriormente classificar.
def criar_pipeline(classificador):
    # Preprocessador + classificador no mesmo Pipeline (fit só no treino)
    return Pipeline([
        ("preprocessador", preprocessador),
        ("classificador", classificador),
    ])


# Garantir uma configuração unica dos modelos dados que não serão alterados
modelos_config = {
    "Regressão Logística": LogisticRegression(max_iter=2000, random_state=SEMENTE),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=SEMENTE),
    "SVM": CalibratedClassifierCV(
        estimator=SVC(
            kernel="rbf",
            class_weight="balanced",
        ),
        method="sigmoid",
        cv=5,
        ensemble=False,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=SEMENTE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=SEMENTE,
        n_jobs=-1,
    )
}

# Realizo o treinamento do dado para todos os modelos mapeados em *modelos_config - Desse modo posso podemos avaliar a melhor opção entre os modelos:
modelos_treinados = {}
for nome, clf in modelos_config.items():
    pipeline_current = criar_pipeline(clf)
    pipeline_current.fit(atributos_treino, rotulo_treino)  # Treino com "Features para treinamento" com "Diagnósticos correspondentes ao treino" 
    modelos_treinados[nome] = pipeline_current
    print(f"{nome}: OK ({len(atributos_treino)} amostras)")

Regressão Logística: OK (455 amostras)
Árvore de Decisão: OK (455 amostras)
SVM: OK (455 amostras)


Gradient Boosting: OK (455 amostras)


Random Forest: OK (455 amostras)


## 4 - Salvar artefato para os notebooks 05 e 06

In [4]:
joblib.dump(
    {
        "modelos_treinados": modelos_treinados,
        "rotulo_teste": rotulo_teste,
        "atributos_teste": atributos_teste,
        "colunas_atributos": colunas_atributos,
    },
    PASTA_ARTEFATOS / "modelos_treinados.joblib",
)
print("Artefato salvo — use nos notebooks 05 e 06")
print(PASTA_ARTEFATOS / "modelos_treinados.joblib")

Artefato salvo — use nos notebooks 05 e 06
E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\04-modelagem-treino\artifacts\modelos_treinados.joblib


## 5 - Lembrete

> **Não avaliar o teste aqui.** Accuracy, recall, F1 e matrizes de confusão ficam no **notebook 05**.
>
> O conjunto de teste foi apenas empacotado no artefato para avaliação honesta depois — sem vazamento de informação no treino.

## 6 - Validação

In [5]:
assert len(modelos_treinados) >= 2
assert (PASTA_ARTEFATOS / "modelos_treinados.joblib").exists()

pacote_saida = joblib.load(PASTA_ARTEFATOS / "modelos_treinados.joblib")
assert set(pacote_saida.keys()) >= {
    "modelos_treinados",
    "rotulo_teste",
    "atributos_teste",
    "colunas_atributos",
}
assert "Regressão Logística" in pacote_saida["modelos_treinados"]
assert "Árvore de Decisão" in pacote_saida["modelos_treinados"]

print("Validação OK")
print("Modelos:", list(pacote_saida["modelos_treinados"].keys()))
print("Artefato:", PASTA_ARTEFATOS / "modelos_treinados.joblib")

Validação OK
Modelos: ['Regressão Logística', 'Árvore de Decisão', 'SVM', 'Gradient Boosting', 'Random Forest']
Artefato: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\04-modelagem-treino\artifacts\modelos_treinados.joblib
